# 03 — Clean and Standardize

Clean geometries, standardize CRS, inspect column names, and prepare join keys.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import (
    RAW_DIR, PROCESSED_DIR, OUTPUT_DIR,
    BOUNDARIES_WFS, HEALTH_WFS, SOCIAL_WFS, POPULATION_CSV,
    METRIC_CRS, MAP_CRS, GEOGRAPHIC_CRS,
    BOUNDARY_LAYER, HEALTH_LAYER, SOCIAL_LAYER,
    MUNICIPALITY_CODE_COL, MUNICIPALITY_NAME_COL, TOTAL_POP_COL,
)
from scripts.data_sources import SOURCES
from scripts.wfs_utils import discover_wfs_layers, load_wfs_layer, download_csv, save_geodataframe, save_dataframe
from scripts.population_utils import normalize_columns, build_population_65_plus
from scripts.analysis_utils import standardize_geodataframes, build_vulnerability_index, spatial_autocorrelation, top_ranked
from scripts.plotting_utils import save_choropleth
from scripts.export_utils import export_geodataframe, export_dataframe

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
boundaries = gpd.read_file(RAW_DIR / "boundaries.gpkg", layer="boundaries")
health = gpd.read_file(RAW_DIR / "health.gpkg", layer="health")
social = gpd.read_file(RAW_DIR / "social.gpkg", layer="social")
pop_raw = pd.read_csv(RAW_DIR / "population_56961.csv", sep=";", encoding="latin1")

boundaries, health, social = standardize_geodataframes(boundaries, health, social, crs=METRIC_CRS)
print(boundaries.crs, health.crs, social.crs)
print(boundaries.shape, health.shape, social.shape)

In [ ]:
print("Boundary columns:")
print(boundaries.columns.tolist())
print("\nPopulation columns:")
print([str(c) for c in pop_raw.columns.tolist()])

In [ ]:
pop_65 = build_population_65_plus(pop_raw)
print(pop_65.head())
print(pop_65.columns.tolist())

In [ ]:
boundaries.to_file(PROCESSED_DIR / "boundaries_clean.gpkg", layer="boundaries", driver="GPKG")
health.to_file(PROCESSED_DIR / "health_clean.gpkg", layer="health", driver="GPKG")
social.to_file(PROCESSED_DIR / "social_clean.gpkg", layer="social", driver="GPKG")
pop_65.to_csv(PROCESSED_DIR / "population_65_plus.csv", index=False)